In [24]:
import pandas as pd
import re, unicodedata

In [25]:
path_to_candidate = r"C:\Users\joly-\Github\HUMAN\tweede_kamer\combined_tk_AI_bert_trial_567.csv"
path_to_labels = r"C:\Users\joly-\Github\HUMAN\tweede_kamer\topics_overview_AI_tk_trial_567.xlsx"
path_to_new_csv = "labelled_TK.csv"

In [26]:
labels = pd.read_excel(path_to_labels)
labels.tail()

,Topic,Count,Name,Representation,Representative_Docs,Representation_Top_30,Label,Meta,Label NL,Meta NL
11,11,24,11_cid_motie_wetsvoorstel_groenlinks,"['cid', 'motie', 'wetsvoorstel', 'groenlinks',...",['onderdeel personeel_materieel begroting defe...,"cid, motie, wetsvoorstel, groenlinks, orde, pv...",political parties,POLITICS & LAW,politieke partijen,POLITIEK & RECHT
12,12,27,12_verfijning_commissie_box_zaak,"['verfijning', 'commissie', 'box', 'zaak', 'pl...",['reactie artificial vooruitgang paar elitewet...,"verfijning, commissie, box, zaak, plaats, forf...",algorithms and social media,SOCIAL MEDIA,algoritmen en sociale media,SOCIALE MEDIA
13,13,35,13_digitalisering_applicatie_transitie_systeem,"['digitalisering', 'applicatie', 'transitie', ...",['desinformatie medium instrument idee realite...,"digitalisering, applicatie, transitie, systeem...",digitalisation and risks,AI RISKS & ETHICS,digitalisering en risico's,AI RISICO'S & ETHIEK
14,14,15,14_toeslag_burger_verrekening_handhavingsstrat...,"['toeslag', 'burger', 'verrekening', 'handhavi...",['openbaarmaking cdio/o&t innovatie cdio/o&t m...,"toeslag, burger, verrekening, handhavingsstrat...",allowances and enforcement,POLITICS & LAW,toeslagen en handhaving,POLITIEK & RECHT
15,15,24,15_dertigledendebat_initiatiefnota_ongelijkhei...,"['dertigledendebat', 'initiatiefnota', 'ongeli...",['rapport zomerreces besluit inhoud onderzoeks...,"dertigledendebat, initiatiefnota, ongelijkheid...",inequality and risks,AI RISKS & ETHICS,ongelijkheid en risico's,AI RISICO'S & ETHIEK


In [27]:
df = pd.read_csv(path_to_candidate, index_col=0)
df.tail()

,title,year,ai_related,company_hits,body,matched_keywords_all,type,persons,orgs,countries,processed,nouns,adjectives,verbs,topic,probability
843,NaN,2024,yes,['x'],De voorzitter:Ik stel voor conform het voorste...,"['x', 'drone', 'drones']",plenair verslag,"['Timmermans', 'Bontenbal', 'Poetin']","['VVD', 'ChristenUnie', 'Trump', 'PVV', 'Kamer...","['VS', 'Lviv', 'Rusland', 'Kyiv', 'West-Europa...",voorstel presidium jaarverslag slotwet defensi...,voorstel presidium jaarverslag_slotwet defensi...,commercieel militair zichtbaar onzichtbaar alg...,stel besluiten besluiten indienen weten rappor...,0,0.387699
844,NaN,2024,yes,['x'],U loopt er steeds voor weg. U houdt hier een m...,"['x', 'claude', 'drones']",plenair verslag,"['Jean-Claude Juncker', 'heer Vermeer', 'Trump...","['PVV', 'JD Vance', 'NSC', 'Netanyahu', 'X', '...","['USA', 'EU', 'Iran', 'Europa', 'Israël', 'VS']",verhaal tijd leugen verraad klasse mens vertro...,verhaal leugen verraad klasse mens vertrouwen ...,mooi grof nodig heel nodig kapot goed heel bel...,lopen houden lopen zeggen tegengewerken werken...,0,0.252044
845,NaN,2024,yes,['x'],"Zij krijgt nr. 1315 (33529). De Kamer,\ngehoor...","['x', 'ai', 'drones']",plenair verslag,"['32802-109', 'heer Holman', 'Mevrouw Postma',...","['31265-129', '31524-621', 'Kamer', 'Europese ...","['Luxemburg', 'Rusland', 'Kyiv', 'Europeesrech...",beraadslaging regering komst aifabriek aanvraa...,beraadslaging regering komst aifabriek aanvraa...,europees ruim aanstaand fysiek voldoende natio...,krijgen horen constateren inspanen overwegenen...,0,0.390809
846,NaN,2024,yes,[],Dan denk ik: het is ook marktwerking; de schol...,"['ai', 'chatgpt']",plenair verslag,['Mevrouw Martens'],[],[],marktwerking school student baan toekomst ai-o...,marktwerking school student baan toekomst ai-o...,heel goed primair eigenlijk,denken bieden hopen willen studeren denken wil...,7,0.062437
847,NaN,2024,yes,"['asml', 'facebook', 'google', 'twitter', 'x']",Dat eerste amendement van mij gaat specifiek o...,"['asml', 'facebook', 'google', 'twitter', 'x',...",plenair verslag,"['heer Krul', 'Trump', 'AliExpress', 'mevrouw ...","['PartyPeeps2000', 'TikTok', 'Europese Commiss...","['geflagged', 'Turkije', 'EU', 'onlinekindermi...",amendement aanbevelingsalgoritme optiek ap dir...,amendement aanbevelingsalgoritme optiek direct...,specifiek apart geschikt natuurlijk bred sec e...,gaan zien toetsen kijken gaan toevoegen gaan p...,5,0.051429


In [28]:
# #inspect topic probabilities
# print("Min probability :", df['probability'].min())
# print("Max probability :", df['probability'].max())
# print("Mean probability:", df['probability'].mean())
# print("Std deviation   :", df['probability'].std())


In [29]:
def norm_topic(x):
    """Canonicalise topic ids for safe matching across df and Excel."""
    if pd.isna(x):
        return None
    s = str(x)
    # unify unicode + strip invisible/space noise
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", "", s)

    # if it looks numeric, coerce to a canonical string:
    #  - 10.0 -> "10"
    #  - 10.50 -> "10.5"
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        else:
            # normalise trimming trailing zeros/dot
            s2 = ("%.15g" % f)  # compact float repr without scientific if possible
            return s2
    except ValueError:
        # non-numeric topic ids like 'FD50' are kept as-is (case-sensitive)
        return s.strip()

# --- 1) Load mapping and normalise its index ---
mapping = pd.read_excel(path_to_labels, index_col=0)
mapping.index = mapping.index.map(norm_topic)

# Optionally: keep only rows that actually have a label
mapping_labeled = mapping[~mapping["Label NL"].isna()].copy()

# --- 2) Normalise df topics too ---
df["topic_norm"] = df["topic"].map(norm_topic)

# --- 3) Build maps and assign ---
label_map_nl = mapping_labeled["Label NL"]
meta_map_nl  = mapping_labeled["Meta NL"]

label_map = mapping_labeled["Label"]
meta_map  = mapping_labeled["Meta"]

df["topic_label_nl"] = df["topic_norm"].map(label_map_nl)
df["topic_meta_nl"]  = df["topic_norm"].map(meta_map_nl)

df["topic_label"] = df["topic_norm"].map(label_map)
df["topic_meta"]  = df["topic_norm"].map(meta_map)

# --- 4) (Optional) sanity checks ---
# What fraction matched?
matched_frac = df["topic_label"].notna().mean()
print(f"Matched labels for {matched_frac:.1%} of rows.")

# If you want to see why some didn’t match, compare key sets:
missing_keys = set(df["topic_norm"].dropna().unique()) - set(mapping_labeled.index)
if missing_keys:
    print(f"{len(missing_keys)} df topic ids not found in mapping (showing up to 20):",
          sorted(list(missing_keys))[:20])

# --- 5) Keep only labelled rows (discard unlabelled) ---
df_labeled = df[df["topic_label"].notna()].copy()
# If you’re done with the helper column:
# df_labeled.drop(columns=["topic_norm"], inplace=True)


Matched labels for 99.9% of rows.
1 df topic ids not found in mapping (showing up to 20): ['-1']


In [30]:
# show rows where topic_label is 'NOISE'
noise_rows = df_labeled[df_labeled['topic_label'] == 'NOISE']
print(f"Rows labeled as 'NOISE': {len(noise_rows)}")

# show row numbers of rows labeled as 'NOISE'
print("Row numbers of 'NOISE' rows:", noise_rows.index.tolist())

# print one row of 'NOISE' rows to inspect
if not noise_rows.empty:
    print("Example 'NOISE' row:")
    print(noise_rows.iloc[0])
    


Rows labeled as 'NOISE': 0
Row numbers of 'NOISE' rows: []


In [315]:
# df.head()

In [31]:
df = df.dropna(subset=['topic_label']).copy()
df = df[df['topic_meta'] != 'NOISE'].reset_index(drop=True)

print(f"rows dropped due to missing labels: {len(df) - len(df_labeled)}")




rows dropped due to missing labels: 0


In [32]:
df.head()

,title,year,ai_related,company_hits,body,matched_keywords_all,type,persons,orgs,countries,...,nouns,adjectives,verbs,topic,probability,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta
0,Investeren in Perspectief (Beleidsnota 2018),2018,yes,"['adyen', 'google', 'x']",Ook biedt de agenda kansen aan het bedrijfslev...,"['adyen', 'google', 'x', 'kunstmatige intellig...",beleidsnota,"['Booking', 'Reinout van den Bergh']","['FMO', '42,8%', 'Google', '41,4%', 'Regeerakk...","['Azië', 'Noord-Amerika', 'Midden-Oosten', 'La...",...,agenda kans bedrijfsleven oplossing digitalise...,innovatief traditioneel groot nieuw universeel...,bieden kennen zet ontstaan ontwikkelen gaan zo...,1,0.279240,1,tech en innovatie,TECH & AI ONTWIKKELING,tech and innovation,TECH & AI DEVELOPMENT
1,Nota Defensie Industrie Strategie,2018,yes,[],Nederland wil zelf aan militaire kennisontwikk...,"['ai', 'artificiële intelligentie', 'drones', ...",beleidsnota,['Meeontwikkelen'],"['GEWENST', 'Energy Wapens', 'Network', 'Human...","['meeontwikkelen', 'Materiel', 'Nederland']",...,kennisontwikkeling analyse kennis prestatie ke...,militair militair artificieel electromagnetisc...,blijven blijven maken ontwikkelen verbeteren w...,1,0.413294,1,tech en innovatie,TECH & AI ONTWIKKELING,tech and innovation,TECH & AI DEVELOPMENT
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021,yes,['google'],Zo wordt in de vernieuwde strategie nu ook de ...,"['google', 'ai', 'algoritmes', 'artificiële in...",beleidsnota,"['Digi-', 'B. I']","['CLARIAH', 'Nederlands Dans Theater', 'Google...",['taalverwerkingstoepassingen'],...,strategie elatie erfgoed kunstensector industr...,creatief nieuw breed nieuw artificieel individ...,vernieuwen leggen bieden geven verkenen realis...,1,0.220815,1,tech en innovatie,TECH & AI ONTWIKKELING,tech and innovation,TECH & AI DEVELOPMENT
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021,yes,['x'],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,"['x', 'ai']",beleidsnota,"['Bl^w', 'voordit', 'Maria 10', 'qaan', 'budge...","['Ic', 'wuo', 'HVP', '03 rendementsklasse', 'T...","['Eengenotsrechttegeneen', 'Septemberbriefover...",...,vierhoek bespreking box d.d. uitstel stelsel b...,politiek nota-stasfb-nota nota-min-memo toekom...,onroerend bespreken onderzoeken bestaan inform...,2,0.127124,2,belasting en vermogen,BEDRIJF & FINANCIËN,tax and assets,BUSINESS & FINANCE
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021,yes,['x'],Dit betreft de nota’s in de onderstaande tabel...,"['x', 'ai']",beleidsnota,"['datvoor de huidige', 'Stas T', 'Teonderteken...","['Unie', 'Europese Commissie', 'afwikkelfonds'...","['Kopleaan', 'veriagen', 'Eventueei', 'Middenb...",...,nota tabel titel aanpak verband vervolgnota ve...,onderstaand herziene abusievelijk vertrouwelij...,betreffen belastingschulden belastingschulen b...,2,0.204630,2,belasting en vermogen,BEDRIJF & FINANCIËN,tax and assets,BUSINESS & FINANCE


In [318]:
# overwrite robots & educatie (3) to 1) robots adn 2) tech & ai ontwikkeling

In [319]:
# df['topic_label'].value_counts()

In [320]:
mask = (
    (df["topic_label"] == "robots en educatie") &
    (df["topic_meta"] == "EDUCATIE")
)

df.loc[mask, "topic_label"] = "robots"
df.loc[mask, "topic_meta"] = "TECH & AI ONTWIKKELING"

In [321]:
# df.shape

In [322]:
df.to_csv(path_to_new_csv)

In [ ]:
# # combine all csv in final_df folder to one csv
# import glob
# all_files = glob.glob("final_df/*.csv")
# df_list = []
# for filename in all_files:
#     df_temp = pd.read_csv(filename, index_col=0)
#     df_list.append(df_temp)
# final_df = pd.concat(df_list, ignore_index=True)

 

In [ ]:
# final_df.shape

(13231, 26)

In [33]:
df['topic_meta_nl'].value_counts()
# show number of topics
print(f"Number of unique topics in 'topic_meta_nl': {df['topic_meta'].nunique()}")

Number of unique topics in 'topic_meta_nl': 10


In [ ]:
df.drop(columns=['label', 'meta', 'label_nl', 'meta_nl'], inplace=True)

In [34]:
df.tail()

,title,year,ai_related,company_hits,body,matched_keywords_all,type,persons,orgs,countries,...,nouns,adjectives,verbs,topic,probability,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta
838,NaN,2024,yes,['x'],De voorzitter:Ik stel voor conform het voorste...,"['x', 'drone', 'drones']",plenair verslag,"['Timmermans', 'Bontenbal', 'Poetin']","['VVD', 'ChristenUnie', 'Trump', 'PVV', 'Kamer...","['VS', 'Lviv', 'Rusland', 'Kyiv', 'West-Europa...",...,voorstel presidium jaarverslag_slotwet defensi...,commercieel militair zichtbaar onzichtbaar alg...,stel besluiten besluiten indienen weten rappor...,0,0.387699,0,leger en drones,AI OORLOGSVOERING EN MILITAIRE CONFLICTEN,military and drones,AI WARFARE & MILIRTARY CONFLICTS
839,NaN,2024,yes,['x'],U loopt er steeds voor weg. U houdt hier een m...,"['x', 'claude', 'drones']",plenair verslag,"['Jean-Claude Juncker', 'heer Vermeer', 'Trump...","['PVV', 'JD Vance', 'NSC', 'Netanyahu', 'X', '...","['USA', 'EU', 'Iran', 'Europa', 'Israël', 'VS']",...,verhaal leugen verraad klasse mens vertrouwen ...,mooi grof nodig heel nodig kapot goed heel bel...,lopen houden lopen zeggen tegengewerken werken...,0,0.252044,0,leger en drones,AI OORLOGSVOERING EN MILITAIRE CONFLICTEN,military and drones,AI WARFARE & MILIRTARY CONFLICTS
840,NaN,2024,yes,['x'],"Zij krijgt nr. 1315 (33529). De Kamer,\ngehoor...","['x', 'ai', 'drones']",plenair verslag,"['32802-109', 'heer Holman', 'Mevrouw Postma',...","['31265-129', '31524-621', 'Kamer', 'Europese ...","['Luxemburg', 'Rusland', 'Kyiv', 'Europeesrech...",...,beraadslaging regering komst aifabriek aanvraa...,europees ruim aanstaand fysiek voldoende natio...,krijgen horen constateren inspanen overwegenen...,0,0.390809,0,leger en drones,AI OORLOGSVOERING EN MILITAIRE CONFLICTEN,military and drones,AI WARFARE & MILIRTARY CONFLICTS
841,NaN,2024,yes,[],Dan denk ik: het is ook marktwerking; de schol...,"['ai', 'chatgpt']",plenair verslag,['Mevrouw Martens'],[],[],...,marktwerking school student baan toekomst ai-o...,heel goed primair eigenlijk,denken bieden hopen willen studeren denken wil...,7,0.062437,7,onderwijs,ONDERWIJS,education,EDUCATION
842,NaN,2024,yes,"['asml', 'facebook', 'google', 'twitter', 'x']",Dat eerste amendement van mij gaat specifiek o...,"['asml', 'facebook', 'google', 'twitter', 'x',...",plenair verslag,"['heer Krul', 'Trump', 'AliExpress', 'mevrouw ...","['PartyPeeps2000', 'TikTok', 'Europese Commiss...","['geflagged', 'Turkije', 'EU', 'onlinekindermi...",...,amendement aanbevelingsalgoritme optiek direct...,specifiek apart geschikt natuurlijk bred sec e...,gaan zien toetsen kijken gaan toevoegen gaan p...,5,0.051429,5,begroting en uitgaven,POLITIEK & RECHT,budget and expenses,POLITICS & LAW


In [36]:
df.to_csv('tweede_kamer_with_topics.csv')